# Support Ticket Triage with Guardrails Part_C

## Mock LLM

In [1]:
"""mock_llm.py -- deterministic, offline stand-in for a hosted LLM chat API (no key, no network, no cost).
Interface mirrors a hosted chat API:  response = chat(messages, tools=None)
    response.choices[0].message.content / .tool_calls ; response.choices[0].finish_reason ; response.usage
Three behaviours are selected by a marker word in the FIRST (system) message:
    "CLASSIFY"  -> support-ticket classifier       "POLICY ASSISTANT" -> grounded policy answerer
    "AGENT"     -> refund agent that requests tools (ReAct style)"""
import json, math, re
from types import SimpleNamespace as NS

class TransientLLMError(Exception):
    """Simulated temporary failure (timeout / server busy). Retrying the same call succeeds."""

_state = {"calls": 0, "ok": 0}      # call counters that drive the simulated misbehaviour


def reset():
    """Restart the counters. Call once before each independent batch run."""
    _state["calls"] = 0; _state["ok"] = 0

CATS = ["billing", "delivery", "technical", "fraud"]
KW = {"fraud": {"unauthorized", "fraud", "hacked", "stolen"},
      "billing": {"charged", "invoice", "bill", "refund", "deducted", "payment"},
      "technical": {"app", "crash", "login", "otp", "error", "freezes"},
      "delivery": {"arrived", "rider", "delivery", "delivered", "late"}}

def est_tokens(text):
    return math.ceil(len((text or "").split()) * 1.3)

def _words(t):
    return set(re.findall(r"[a-z]+", (t or "").lower()))

def _resp(content=None, tool_calls=None, finish="stop", messages=()):
    prompt = sum(est_tokens(m.get("content") or json.dumps(m.get("tool_calls", ""), default=str)) for m in messages)
    comp = est_tokens(content or json.dumps(tool_calls or "", default=str))
    msg = NS(role="assistant", content=content, tool_calls=[NS(**tc) for tc in (tool_calls or [])])
    return NS(choices=[NS(message=msg, finish_reason=finish)],
              usage=NS(prompt_tokens=prompt, completion_tokens=comp, total_tokens=prompt + comp))

def _classify(messages):
    system, user = messages[0]["content"], messages[-1]["content"]
    _state["calls"] += 1
    if _state["calls"] % 9 == 0:                                 # every 9th call fails once
        raise TransientLLMError("503 server busy")
    _state["ok"] += 1
    order = ["fraud", "technical", "billing", "delivery"] if "EXAMPLES:" in system else ["billing", "fraud", "technical", "delivery"]
    w = _words(user)
    cat = next((c for c in order if KW[c] & w), "technical")
    if _state["ok"] % 13 == 0:                                   # chatty, non-JSON answer
        return _resp(f"The customer appears to have a {cat}-related problem.", messages=messages)
    if _state["ok"] % 17 == 0:                                   # JSON, but category outside the allowed list
        return _resp(json.dumps({"category": "shipping", "reason": "parcel movement"}), messages=messages)
    return _resp(json.dumps({"category": cat, "reason": "keyword match"}), messages=messages)

STOP = {"the", "a", "an", "is", "of", "to", "for", "in", "on", "how", "what", "when", "who", "are", "do", "does",
        "i", "my", "be", "can", "after", "from", "it", "or", "and", "at", "by", "which", "why", "if", "than", "that"}

def _stems(t):
    return {w[:5] for w in re.findall(r"[a-z]+", (t or "").lower()) if w not in STOP}

def _policy(messages):
    """Answers with the context sentence that best matches the question (rare words weigh more)."""
    user = messages[-1]["content"]
    m = re.search(r"CONTEXT:\s*(.*?)\s*QUESTION:\s*(.*)", user, re.S)
    context, question = (m.group(1), m.group(2)) if m else ("", user)
    sents = [s.strip() for s in re.split(r"(?<=[.])\s+", re.sub(r"\[[^\]]*\.txt\]", " ", context)) if s.strip()]
    qs = _stems(question)
    df = {q: sum(q in _stems(s) for s in sents) for q in qs}
    best, best_score = None, 0.0
    for s in sents:
        sc = sum(1.0 / df[q] for q in qs & _stems(s))
        if sc > best_score:
            best, best_score = s, sc
    if best is None:
        return _resp("I could not find that information in the policy documents.", messages=messages)
    return _resp(best, messages=messages)

def _agent(messages, tools):
    user = next(m["content"] for m in messages if m["role"] == "user")
    tool_msgs = [m for m in messages if m["role"] == "tool"]
    call = lambda n, a: _resp(None, [{"id": f"call_{len(tool_msgs)+1}", "name": n, "arguments": a}], "tool_calls", messages)
    if "[LOOP]" in user:
        return call("search_policy", {"query": "refund policy"})
    if not tool_msgs:
        m = re.search(r"ORD-\d{6}", user)
        return call("get_order", {"order_id": m.group(0)}) if m else _resp("NEEDS_INFO: please share the order id.", messages=messages)
    last = tool_msgs[-1]; data = json.loads(last["content"]); name = last["name"]
    if name == "get_order":
        if "error" in data:
            return _resp("NOT_FOUND: no such order.", messages=messages)
        return call("search_policy", {"query": "refund policy late delivery"})
    order = json.loads(next(m for m in tool_msgs if m["name"] == "get_order")["content"])
    if name == "search_policy":
        if order["status"] != "completed":
            return _resp("DENIED: only completed orders can be refunded.", messages=messages)
        return call("calculate_refund", {"order_amount": order["order_amount"], "delivery_minutes": order["delivery_minutes"]})
    if name == "calculate_refund":
        amt = data["refund_inr"]
        if amt == 0:
            return _resp("DENIED: delivery was within 45 minutes.", messages=messages)
        if amt > 1000:
            return call("escalate_to_human", {"order_id": order["order_id"], "refund_inr": amt})
        return call("issue_refund", {"order_id": order["order_id"], "refund_inr": amt})
    if name == "issue_refund":
        return _resp(f"REFUND: INR {data['refund_inr']} issued.", messages=messages)
    if name == "escalate_to_human":
        return _resp("ESCALATE: refund above INR 1000 sent to a human lead.", messages=messages)
    return _resp("DENIED: unable to process.", messages=messages)

def chat(messages, tools=None):
    head = messages[0]["content"].upper()
    if "CLASSIFY" in head:
        return _classify(messages)
    if "POLICY ASSISTANT" in head:
        return _policy(messages)
    if "AGENT" in head:
        return _agent(messages, tools)
    raise ValueError("first message must name one of: CLASSIFY, POLICY ASSISTANT, AGENT")


In [2]:
test_text = "My order has not arrived yet. Please help me."

token_count = est_tokens(test_text)

print("Test text:", test_text)
print("Estimated tokens:", token_count)

Test text: My order has not arrived yet. Please help me.
Estimated tokens: 12


In [3]:
# Reset the mock model before testing
reset()

# Create a test prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are a support ticket classifier. "
            "CLASSIFY the ticket into exactly one category: "
            "billing, delivery, technical, fraud. "
            'Return only JSON: {"category": "...", "reason": "..."}'
        )
    },
    {
        "role": "user",
        "content": "My order arrived late and has not been delivered."
    }
]

# Call the mock model
response = chat(messages)

# Display the response
print("Model response:")
print(response.choices[0].message.content)

print("\nTotal tokens used:")
print(response.usage.total_tokens)

Model response:
{"category": "delivery", "reason": "keyword match"}

Total tokens used:
51


# Tasks

# 1. Input guardrails 

## Load the support tickets

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import random
import time

# Find the main project folder
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Define file paths
TICKETS_PATH = BASE_DIR / "data" / "raw" / "tickets.csv"
OUTPUT_DIR = BASE_DIR / "outputs"

# Create outputs folder if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load support tickets
tickets = pd.read_csv(
    TICKETS_PATH,
    keep_default_na=False
)

print("Tickets loaded successfully!")
print("Dataset shape:", tickets.shape)

display(tickets.head())

Tickets loaded successfully!
Dataset shape: (164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical
3,T0159,C1172,,,none
4,T0080,C0910,ORD-109516,I was charged for order ORD-109516 that I neve...,fraud


In [5]:
tickets

,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical
3,T0159,C1172,,,none
4,T0080,C0910,ORD-109516,I was charged for order ORD-109516 that I neve...,fraud
...,...,...,...,...,...
159,T0150,C0622,ORD-106507,The rider marked ORD-106507 as delivered but I...,delivery
160,T0065,C0205,ORD-102139,The invoice for ORD-102139 shows a wrong amoun...,billing
161,T0128,C0467,ORD-104881,Someone hacked my account and used my wallet b...,fraud
162,T0005,C0404,ORD-104288,"Extra money was deducted for order ORD-104288,...",billing


## 1.2 Check the dataset

In [6]:
# Check column names
print("Columns:", tickets.columns.tolist())

# Check total number of tickets
print("Total tickets:", len(tickets))

# Count empty or whitespace-only ticket texts
empty_count = tickets["text"].astype(str).str.strip().eq("").sum()

print("Empty tickets:", empty_count)

# Estimate tokens for every ticket
tickets["estimated_tokens"] = tickets["text"].apply(est_tokens)

# Count tickets exceeding 400 tokens
long_count = (tickets["estimated_tokens"] > 400).sum()

print("Long tickets:", long_count)

# Display ticket ID and estimated token count
display(
    tickets[["ticket_id", "estimated_tokens"]].head(10)
)

Columns: ['ticket_id', 'customer_id', 'order_id', 'text', 'true_category']
Total tickets: 164
Empty tickets: 4
Long tickets: 4


,ticket_id,estimated_tokens
0,T0024,25
1,T0073,21
2,T0095,23
3,T0159,0
4,T0080,25
5,T0124,25
6,T0061,17
7,T0118,20
8,T0037,21
9,T0002,19


In [7]:
# Create a separate working copy
tickets_processed = tickets.copy()

# Initially, mark every ticket as ready
tickets_processed["status"] = "ready_for_classification"

# Rule 1: Identify empty or whitespace-only tickets
empty_mask = (
    tickets_processed["text"]
    .astype(str)
    .str.strip()
    .eq("")
)

tickets_processed.loc[empty_mask, "status"] = "skipped_empty"


# Rule 2: Identify tickets exceeding 400 tokens
long_mask = tickets_processed["estimated_tokens"] > 400

tickets_processed.loc[
    long_mask & ~empty_mask,
    "status"
] = "flagged_long"


# Display status counts
print("Ticket status counts:")
print(tickets_processed["status"].value_counts())

# Count eligible tickets
eligible_count = (
    tickets_processed["status"] == "ready_for_classification"
).sum()

print("\nEligible tickets:", eligible_count)

# Display a sample
display(
    tickets_processed[
        ["ticket_id", "estimated_tokens", "status"]
    ].head(10)
)

Ticket status counts:
status
ready_for_classification    156
skipped_empty                 4
flagged_long                  4
Name: count, dtype: int64

Eligible tickets: 156


,ticket_id,estimated_tokens,status
0,T0024,25,ready_for_classification
1,T0073,21,ready_for_classification
2,T0095,23,ready_for_classification
3,T0159,0,skipped_empty
4,T0080,25,ready_for_classification
5,T0124,25,ready_for_classification
6,T0061,17,ready_for_classification
7,T0118,20,ready_for_classification
8,T0037,21,ready_for_classification
9,T0002,19,ready_for_classification


# 2. Redact personal information 

In [8]:
# Regular expressions for sensitive information

EMAIL_PATTERN = r"[\w.+-]+@[\w-]+\.[\w.]+"

PHONE_PATTERN = r"\b[6-9]\d{9}\b"

ORDER_ID_PATTERN = r"ORD-\d{6}"

## 2.1 Create a redact_pii funtion

In [9]:
def redact_pii(text):

    text = str(text)

    # 1. Redact email addresses
    text, email_count = re.subn(
        EMAIL_PATTERN,
        "[EMAIL]",
        text
    )

    # 2. Redact phone numbers
    text, phone_count = re.subn(
        PHONE_PATTERN,
        "[PHONE]",
        text
    )

    # 3. Redact order IDs
    text, order_id_count = re.subn(
        ORDER_ID_PATTERN,
        "[ORDER_ID]",
        text
    )

    return text, {
        "email": email_count,
        "phone": phone_count,
        "order_id": order_id_count
    }

## 2.2 Sample email test for pii

In [10]:
sample_text = "Contact me at praveenlk123@gmail.com"

redacted_text, counts = redact_pii(sample_text)

print(redacted_text)
print(counts)

Contact me at [EMAIL]
{'email': 1, 'phone': 0, 'order_id': 0}


In [11]:
# Apply redaction to every ticket
redaction_results = tickets_processed["text"].apply(redact_pii)

# Store redacted text separately
tickets_processed["redacted_text"] = redaction_results.apply(
    lambda result: result[0]
)

# Calculate total replacements
redaction_counts = {
    "email": sum(result[1]["email"] for result in redaction_results),
    "phone": sum(result[1]["phone"] for result in redaction_results),
    "order_id": sum(result[1]["order_id"] for result in redaction_results)
}

# Display redaction report
print("Redaction Report")
print("----------------")

for item, count in redaction_counts.items():
    print(f"{item}: {count}")

print("Total replacements:", sum(redaction_counts.values()))

Redaction Report
----------------
email: 55
phone: 91
order_id: 132
Total replacements: 278


In [12]:
# Check whether any sensitive information remains

remaining_emails = tickets_processed["redacted_text"].str.contains(
    EMAIL_PATTERN,
    regex=True
).sum()

remaining_phones = tickets_processed["redacted_text"].str.contains(
    PHONE_PATTERN,
    regex=True
).sum()

remaining_order_ids = tickets_processed["redacted_text"].str.contains(
    ORDER_ID_PATTERN,
    regex=True
).sum()

print("Remaining sensitive information:")
print("Emails:", remaining_emails)
print("Phones:", remaining_phones)
print("Order IDs:", remaining_order_ids)

Remaining sensitive information:
Emails: 0
Phones: 0
Order IDs: 0


# 3. Two prompts 

# 3.1 Create a prompt for zero short encoding

In [13]:
ZERO_SHOT_PROMPT = """
You are a customer support ticket classification assistant.

CLASSIFY the customer's ticket into exactly one of these categories:
billing, delivery, technical, fraud.

Return only valid JSON in this exact format:
{"category": "...", "reason": "..."}

Do not include explanations outside the JSON.
"""

## 3.2 Create a prompt for few short encoding

In [14]:
FEW_SHOT_PROMPT = """
You are a customer support ticket classification assistant.

CLASSIFY the customer's ticket into exactly one of these categories:
billing, delivery, technical, fraud.

Return only valid JSON in this exact format:
{"category": "...", "reason": "..."}

Do not include explanations outside the JSON.

EXAMPLES:

Ticket: "I noticed an unauthorized transaction in my account."
JSON: {"category": "fraud", "reason": "Unauthorized transaction reported."}

Ticket: "My order arrived late and the delivery was delayed."
JSON: {"category": "delivery", "reason": "Customer reports a delivery delay."}

Ticket: "The app crashes whenever I try to log in."
JSON: {"category": "technical", "reason": "Customer reports an application problem."}

Ticket: "I was charged twice for the same order."
JSON: {"category": "billing", "reason": "Customer reports a duplicate charge."}
"""

## 3.3 Validate both encoding

In [15]:
# Check zero-shot prompt
assert "CLASSIFY" in ZERO_SHOT_PROMPT
assert "EXAMPLES:" not in ZERO_SHOT_PROMPT

# Check few-shot prompt
assert "CLASSIFY" in FEW_SHOT_PROMPT
assert "EXAMPLES:" in FEW_SHOT_PROMPT

# Check all four categories
categories = ["billing", "delivery", "technical", "fraud"]

for category in categories:
    assert category in ZERO_SHOT_PROMPT
    assert category in FEW_SHOT_PROMPT

print("Zero-shot prompt validation: PASSED")
print("Few-shot prompt validation: PASSED")
print("All four categories are included.")

Zero-shot prompt validation: PASSED
Few-shot prompt validation: PASSED
All four categories are included.


In [16]:
print("Zero-shot prompt tokens:", est_tokens(ZERO_SHOT_PROMPT))

print("Few-shot prompt tokens:", est_tokens(FEW_SHOT_PROMPT))

Zero-shot prompt tokens: 54
Few-shot prompt tokens: 149


# 4. Resilient calls

## 4.1 Create the retry function

In [17]:
TransientLLMError

__main__.TransientLLMError

In [18]:
# Base waiting time
BASE_DELAY = 0.05

# Maximum number of retries
MAX_RETRIES = 3


def call_model_with_retry(ticket_id, prompt, redacted_text):

    # System prompt and user message
    messages = [
        {
            "role": "system",
            "content": prompt
        },
        {
            "role": "user",
            "content": redacted_text
        }
    ]

    # Initial attempt + maximum 3 retries
    for attempt in range(MAX_RETRIES + 1):

        try:
            # Call the mock LLM
            response = chat(messages)

            # Return successful response
            return response

        except TransientLLMError:

            # If all retries are exhausted, raise the error
            if attempt == MAX_RETRIES:
                raise

            # Calculate exponential backoff with jitter
            wait_time = (
                BASE_DELAY
                * (2 ** attempt)
                * random.uniform(0.9, 1.15)
            )

            # Display retry information
            print(
                f"Ticket {ticket_id}: "
                f"Retry {attempt}, "
                f"waiting {wait_time:.4f} seconds"
            )

            # Wait before trying again
            time.sleep(wait_time)

## 4.2 Test the function

In [19]:
# Select one eligible ticket
test_ticket = tickets_processed[
    tickets_processed["status"] == "ready_for_classification"
].iloc[0]

# Reset mock LLM before this standalone test
reset()

# Call the model using the zero-shot prompt
test_response = call_model_with_retry(
    ticket_id=test_ticket["ticket_id"],
    prompt=ZERO_SHOT_PROMPT,
    redacted_text=test_ticket["redacted_text"]
)

# Display response
print("\nModel response:")
print(test_response.choices[0].message.content)

print("\nTokens used:")
print(test_response.usage.total_tokens)


Model response:
{"category": "fraud", "reason": "keyword match"}

Tokens used:
86


# 5. Validate every answer 

In [20]:
ALLOWED_CATEGORIES = [
    "billing",
    "delivery",
    "technical",
    "fraud"
]

In [21]:
def validate_response(response_text):

    # Try to parse the response as JSON
    try:
        result = json.loads(response_text)

        # Check whether the category is allowed
        if (
            isinstance(result, dict)
            and result.get("category") in ALLOWED_CATEGORIES
        ):
            return {
                "category": result["category"],
                "reason": result.get("reason", ""),
                "status": "classified_json"
            }

    except (json.JSONDecodeError, TypeError):
        pass

    # If JSON validation fails, extract lowercase words
    words = re.findall(r"[a-z]+", response_text.lower())

    # Find allowed category words
    matched_categories = [
        word for word in words
        if word in ALLOWED_CATEGORIES
    ]

    # Accept only when exactly one category is found
    if len(matched_categories) == 1:
        return {
            "category": matched_categories[0],
            "reason": "",
            "status": "classified_keyword"
        }

    # Otherwise flag for human review
    return {
        "category": "",
        "reason": "",
        "status": "flagged_invalid_output"
    }

In [22]:
test_response = call_model_with_retry(
    ticket_id=test_ticket["ticket_id"],
    prompt=ZERO_SHOT_PROMPT,
    redacted_text=test_ticket["redacted_text"]
)

In [23]:
response_text = test_response.choices[0].message.content

In [24]:
# Extract the model's response text
response_text = test_response.choices[0].message.content

# Convert JSON string into a Python dictionary
result = json.loads(response_text)

# Display the result
print(result)
print(type(result))

{'category': 'fraud', 'reason': 'keyword match'}
<class 'dict'>


In [25]:
text = "My order has a delivery problem"

words = text.lower().split()

print(words)

['my', 'order', 'has', 'a', 'delivery', 'problem']


In [26]:
matched_categories = [
        word for word in words
        if word in ALLOWED_CATEGORIES
    ]


In [27]:
ALLOWED_CATEGORIES = [
    "billing",
    "delivery",
    "technical",
    "fraud"
]


def validate_response(response_text):

    # Try JSON validation
    try:
        result = json.loads(response_text)

        if (
            isinstance(result, dict)
            and result.get("category") in ALLOWED_CATEGORIES
        ):
            return {
                "category": result["category"],
                "reason": result.get("reason", ""),
                "status": "classified_json"
            }

        # Valid JSON but unsupported structure/category
        return {
            "category": "",
            "reason": "",
            "status": "flagged_invalid_output"
        }

    except (json.JSONDecodeError, TypeError):

        pass

    # Keyword validation
    words = re.findall(r"[a-z]+", response_text.lower())

    matched_categories = [
        word for word in words
        if word in ALLOWED_CATEGORIES
    ]

    if len(matched_categories) == 1:
        return {
            "category": matched_categories[0],
            "reason": "",
            "status": "classified_keyword"
        }

    return {
        "category": "",
        "reason": "",
        "status": "flagged_invalid_output"
    }

In [28]:
result = json.loads(response_text)

In [29]:
result.get("category") in ALLOWED_CATEGORIES

True

In [30]:
words = re.findall(r"[a-z]+", response_text.lower())

In [31]:
if len(matched_categories) == 1:
    print("One category found")

One category found


In [32]:
# Test 1: Valid JSON
test_1 = validate_response(
    '{"category": "billing", "reason": "Payment issue"}'
)

# Test 2: Keyword response
test_2 = validate_response(
    "This ticket belongs to billing."
)

# Test 3: Invalid response
test_3 = validate_response(
    "This ticket belongs to shipping."
)

print("Test 1:", test_1)
print("Test 2:", test_2)
print("Test 3:", test_3)

Test 1: {'category': 'billing', 'reason': 'Payment issue', 'status': 'classified_json'}
Test 2: {'category': 'billing', 'reason': '', 'status': 'classified_keyword'}
Test 3: {'category': '', 'reason': '', 'status': 'flagged_invalid_output'}


In [33]:
# Test 1: Valid JSON
test_1 = validate_response(
    '{"category": "billing", "reason": "Payment issue"}'
)

# Test 2: Chatty response with hyphen
test_2 = validate_response(
    "The customer appears to have a billing-related problem."
)

# Test 3: Two allowed categories
test_3 = validate_response(
    "This is a billing and fraud problem."
)

# Test 4: Valid JSON but unsupported category
test_4 = validate_response(
    '{"category": "shipping", "reason": "Delivery issue"}'
)

# Test 5: Invalid response with no allowed category
test_5 = validate_response(
    "This is an unknown issue."
)

print("Test 1:", test_1)
print("Test 2:", test_2)
print("Test 3:", test_3)
print("Test 4:", test_4)
print("Test 5:", test_5)

Test 1: {'category': 'billing', 'reason': 'Payment issue', 'status': 'classified_json'}
Test 2: {'category': 'billing', 'reason': '', 'status': 'classified_keyword'}
Test 3: {'category': '', 'reason': '', 'status': 'flagged_invalid_output'}
Test 4: {'category': '', 'reason': '', 'status': 'flagged_invalid_output'}
Test 5: {'category': '', 'reason': '', 'status': 'flagged_invalid_output'}


# 6. Compare zero-shot and few-shot 

## 6.1 Create a reusable run_classification function

In [34]:
def run_classification(prompt):

    results = []

    # Process tickets one by one
    for _, row in tickets_processed.iterrows():

        ticket_id = row["ticket_id"]

        # Get the existing input guardrail status
        if row["status"] != "ready_for_classification":

            results.append({
                "ticket_id": ticket_id,
                "customer_id": row["customer_id"],
                "true_category": row["true_category"],
                "category": "",
                "reason": "",
                "status": row["status"],
                "tokens_used": 0
            })

            continue

        # Call the mock LLM with retry logic
        try:
            response = call_model_with_retry(
                ticket_id=ticket_id,
                prompt=prompt,
                redacted_text=row["redacted_text"]
            )

            # Extract model response text
            response_text = response.choices[0].message.content

            # Validate model response
            validation = validate_response(response_text)

            # Store result
            results.append({
                "ticket_id": ticket_id,
                "customer_id": row["customer_id"],
                "true_category": row["true_category"],
                "category": validation["category"],
                "reason": validation["reason"],
                "status": validation["status"],
                "tokens_used": response.usage.total_tokens
            })

        except TransientLLMError:

            # Handle a ticket if all retries fail
            results.append({
                "ticket_id": ticket_id,
                "customer_id": row["customer_id"],
                "true_category": row["true_category"],
                "category": "",
                "reason": "",
                "status": "model_error",
                "tokens_used": 0
            })

    return pd.DataFrame(results)

## 6.2 Zero shot encoding

In [35]:
# Reset mock model immediately before the full run
reset()

# Run zero-shot classification
zero_shot_results = run_classification(
    ZERO_SHOT_PROMPT
)

print("Zero-shot classification completed!")

display(zero_shot_results.head())

Ticket T0002: Retry 0, waiting 0.0488 seconds


Ticket T0116: Retry 0, waiting 0.0483 seconds
Ticket T0153: Retry 0, waiting 0.0542 seconds
Ticket T0039: Retry 0, waiting 0.0507 seconds
Ticket T0152: Retry 0, waiting 0.0483 seconds
Ticket T0043: Retry 0, waiting 0.0458 seconds
Ticket T0111: Retry 0, waiting 0.0472 seconds
Ticket T0155: Retry 0, waiting 0.0465 seconds
Ticket T0100: Retry 0, waiting 0.0472 seconds
Ticket T0068: Retry 0, waiting 0.0480 seconds
Ticket T0135: Retry 0, waiting 0.0575 seconds
Ticket T0009: Retry 0, waiting 0.0554 seconds
Ticket T0067: Retry 0, waiting 0.0451 seconds
Ticket T0106: Retry 0, waiting 0.0457 seconds
Ticket T0097: Retry 0, waiting 0.0486 seconds
Ticket T0003: Retry 0, waiting 0.0536 seconds
Ticket T0019: Retry 0, waiting 0.0542 seconds
Ticket T0086: Retry 0, waiting 0.0560 seconds
Ticket T0065: Retry 0, waiting 0.0574 seconds
Zero-shot classification completed!


,ticket_id,customer_id,true_category,category,reason,status,tokens_used
0,T0024,C0486,fraud,fraud,keyword match,classified_json,86
1,T0073,C0080,billing,billing,keyword match,classified_json,82
2,T0095,C1087,technical,billing,keyword match,classified_json,84
3,T0159,C1172,none,,,skipped_empty,0
4,T0080,C0910,fraud,billing,keyword match,classified_json,86


## 6.3 Zero shot encoding

In [36]:
# Reset mock model immediately before the full run
reset()

# Run few-shot classification
few_shot_results = run_classification(
    FEW_SHOT_PROMPT
)

print("Few-shot classification completed!")

display(few_shot_results.head())

Ticket T0002: Retry 0, waiting 0.0452 seconds
Ticket T0116: Retry 0, waiting 0.0464 seconds
Ticket T0153: Retry 0, waiting 0.0468 seconds
Ticket T0039: Retry 0, waiting 0.0508 seconds
Ticket T0152: Retry 0, waiting 0.0523 seconds
Ticket T0043: Retry 0, waiting 0.0479 seconds
Ticket T0111: Retry 0, waiting 0.0475 seconds
Ticket T0155: Retry 0, waiting 0.0509 seconds
Ticket T0100: Retry 0, waiting 0.0556 seconds
Ticket T0068: Retry 0, waiting 0.0468 seconds
Ticket T0135: Retry 0, waiting 0.0530 seconds
Ticket T0009: Retry 0, waiting 0.0516 seconds
Ticket T0067: Retry 0, waiting 0.0546 seconds
Ticket T0106: Retry 0, waiting 0.0563 seconds
Ticket T0097: Retry 0, waiting 0.0453 seconds
Ticket T0003: Retry 0, waiting 0.0510 seconds
Ticket T0019: Retry 0, waiting 0.0532 seconds
Ticket T0086: Retry 0, waiting 0.0557 seconds
Ticket T0065: Retry 0, waiting 0.0480 seconds
Few-shot classification completed!


,ticket_id,customer_id,true_category,category,reason,status,tokens_used
0,T0024,C0486,fraud,fraud,keyword match,classified_json,181
1,T0073,C0080,billing,billing,keyword match,classified_json,177
2,T0095,C1087,technical,technical,keyword match,classified_json,179
3,T0159,C1172,none,,,skipped_empty,0
4,T0080,C0910,fraud,fraud,keyword match,classified_json,181


In [37]:
print("ZERO-SHOT STATUS COUNTS")
print(zero_shot_results["status"].value_counts())
print("\n TOTAL OF ZERO-SHOT STATUS COUNTS : ")
print(zero_shot_results["status"].value_counts().sum())

print("\nFEW-SHOT STATUS COUNTS")
print(few_shot_results["status"].value_counts())
print("\n TOTAL OF FEW-SHOT STATUS COUNTS : ")
print(zero_shot_results["status"].value_counts().sum())

ZERO-SHOT STATUS COUNTS
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
flagged_long                4
skipped_empty               4
Name: count, dtype: int64

 TOTAL OF ZERO-SHOT STATUS COUNTS : 
164

FEW-SHOT STATUS COUNTS
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
flagged_long                4
skipped_empty               4
Name: count, dtype: int64

 TOTAL OF FEW-SHOT STATUS COUNTS : 
164


In [38]:
# Reset mock LLM before the full run
reset()

# Run zero-shot classification
zero_shot_results = run_classification(
    ZERO_SHOT_PROMPT
)

print("Zero-shot classification completed!")

# Display first five results
display(zero_shot_results.head())

Ticket T0002: Retry 0, waiting 0.0531 seconds
Ticket T0116: Retry 0, waiting 0.0551 seconds
Ticket T0153: Retry 0, waiting 0.0463 seconds
Ticket T0039: Retry 0, waiting 0.0460 seconds
Ticket T0152: Retry 0, waiting 0.0477 seconds
Ticket T0043: Retry 0, waiting 0.0466 seconds
Ticket T0111: Retry 0, waiting 0.0530 seconds
Ticket T0155: Retry 0, waiting 0.0507 seconds
Ticket T0100: Retry 0, waiting 0.0482 seconds
Ticket T0068: Retry 0, waiting 0.0467 seconds
Ticket T0135: Retry 0, waiting 0.0538 seconds
Ticket T0009: Retry 0, waiting 0.0455 seconds
Ticket T0067: Retry 0, waiting 0.0526 seconds
Ticket T0106: Retry 0, waiting 0.0513 seconds
Ticket T0097: Retry 0, waiting 0.0529 seconds
Ticket T0003: Retry 0, waiting 0.0500 seconds
Ticket T0019: Retry 0, waiting 0.0561 seconds
Ticket T0086: Retry 0, waiting 0.0455 seconds
Ticket T0065: Retry 0, waiting 0.0485 seconds
Zero-shot classification completed!


,ticket_id,customer_id,true_category,category,reason,status,tokens_used
0,T0024,C0486,fraud,fraud,keyword match,classified_json,86
1,T0073,C0080,billing,billing,keyword match,classified_json,82
2,T0095,C1087,technical,billing,keyword match,classified_json,84
3,T0159,C1172,none,,,skipped_empty,0
4,T0080,C0910,fraud,billing,keyword match,classified_json,86


In [39]:
print("ZERO-SHOT")
print("Total rows:", len(zero_shot_results))
print("Total tokens:", zero_shot_results["tokens_used"].sum())

print("\nFEW-SHOT")
print("Total rows:", len(few_shot_results))
print("Total tokens:", few_shot_results["tokens_used"].sum())

ZERO-SHOT
Total rows: 164
Total tokens: 12832

FEW-SHOT
Total rows: 164
Total tokens: 27652


In [40]:
def calculate_metrics(results):

    # Select successfully classified tickets
    classified = results[
        results["status"].isin([
            "classified_json",
            "classified_keyword"
        ])
    ]

    # Calculate accuracy
    accuracy = (
        classified["category"] == classified["true_category"]
    ).mean() * 100

    # Count classified tickets
    classified_count = len(classified)

    # Calculate total token usage
    total_tokens = results["tokens_used"].sum()

    # Calculate tokens per classified ticket
    tokens_per_ticket = (
        total_tokens / classified_count
        if classified_count > 0 else 0
    )

    return {
        "tickets_classified": classified_count,
        "accuracy": accuracy,
        "total_tokens": total_tokens,
        "tokens_per_classified_ticket": tokens_per_ticket
    }

In [41]:
# Calculate zero-shot metrics
zero_metrics = calculate_metrics(zero_shot_results)

# Calculate few-shot metrics
few_metrics = calculate_metrics(few_shot_results)

# Create comparison table
comparison = pd.DataFrame(
    [zero_metrics, few_metrics],
    index=["Zero-shot", "Few-shot"]
)

display(comparison.round(2))

,tickets_classified,accuracy,total_tokens,tokens_per_classified_ticket
Zero-shot,147,85.71,12832,87.29
Few-shot,147,100.00,27652,188.11


In [42]:
# Select successfully classified zero-shot tickets
zero_classified = zero_shot_results[
    zero_shot_results["status"].isin([
        "classified_json",
        "classified_keyword"
    ])
]

# Create confusion table
confusion_table = pd.crosstab(
    zero_classified["true_category"],
    zero_classified["category"],
    rownames=["Actual"],
    colnames=["Predicted"],
    dropna=False
)

display(confusion_table)

Predicted,billing,delivery,fraud,technical
Actual,,,,
billing,38,0,0,0
delivery,0,36,0,0
fraud,11,0,23,0
technical,10,0,0,29


In [43]:
# Check both experiment datasets
for name, results in [
    ("ZERO-SHOT", zero_shot_results),
    ("FEW-SHOT", few_shot_results)
]:

    classified = results[
        results["status"].isin([
            "classified_json",
            "classified_keyword"
        ])
    ]

    accuracy = (
        classified["category"] == classified["true_category"]
    ).mean() * 100

    total_tokens = results["tokens_used"].sum()

    tokens_per_ticket = (
        total_tokens / len(classified)
        if len(classified) > 0 else 0
    )

    print("\n", name)
    print("-------------------------")
    print("Total tickets:", len(results))
    print("Classified tickets:", len(classified))
    print("Accuracy:", round(accuracy, 2), "%")
    print("Total tokens:", total_tokens)
    print("Tokens per classified ticket:", round(tokens_per_ticket, 2))

    print("\nStatus counts:")
    print(results["status"].value_counts())


 ZERO-SHOT
-------------------------
Total tickets: 164
Classified tickets: 147
Accuracy: 85.71 %
Total tokens: 12832
Tokens per classified ticket: 87.29

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
flagged_long                4
skipped_empty               4
Name: count, dtype: int64

 FEW-SHOT
-------------------------
Total tickets: 164
Classified tickets: 147
Accuracy: 100.0 %
Total tokens: 27652
Tokens per classified ticket: 188.11

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
flagged_long                4
skipped_empty               4
Name: count, dtype: int64


In [44]:
zero_classified = zero_shot_results[
    zero_shot_results["status"].isin([
        "classified_json",
        "classified_keyword"
    ])
]

confusion_table = pd.crosstab(
    zero_classified["true_category"],
    zero_classified["category"]
).reindex(
    index=["billing", "delivery", "technical", "fraud"],
    columns=["billing", "delivery", "technical", "fraud"],
    fill_value=0
)

display(confusion_table)

category,billing,delivery,technical,fraud
true_category,,,,
billing,38,0,0,0
delivery,0,36,0,0
technical,10,0,29,0
fraud,11,0,0,23


# 7. Budget, routing and output 

 (a) Add a TOKEN_BUDGET: before each model call, if the running total of usage.total_tokens is already ≥ the budget, make no call and set status budget_exhausted (route to human review). Run the zero-shot pipeline with TOKEN_BUDGET = 6000 and save the result as outputs/triage_budget_demo.csv 
 at least one ticket must end budget_exhausted, and you report how many. If your prompt is so short that no ticket is exhausted, lower the demo budget (for example to 5000) and report the budget you used. The count depends on your prompt length (for example 51 at about 29 estimated tokens and 58 at about 33); only "at least one ticket exhausted and a reported count" is graded. 

In [45]:
TOKEN_BUDGET = 6000

In [46]:
tokens_used = response.usage.total_tokens
total_tokens += tokens_used

In [47]:
def run_classification_with_budget(prompt, TOKEN_BUDGET):

    results = []

    # Running total of successfully returned response tokens
    total_tokens = 0

    # Process all tickets
    for _, row in tickets_processed.iterrows():

        ticket_id = row["ticket_id"]

        # First check the existing input guardrails
        if row["status"] != "ready_for_classification":

            results.append({
                "ticket_id": ticket_id,
                "customer_id": row["customer_id"],
                "true_category": row["true_category"],
                "category": "",
                "reason": "",
                "status": row["status"],
                "tokens_used": 0
            })

            continue

        # Check token budget BEFORE calling the model
        if total_tokens >= TOKEN_BUDGET:

            results.append({
                "ticket_id": ticket_id,
                "customer_id": row["customer_id"],
                "true_category": row["true_category"],
                "category": "",
                "reason": "",
                "status": "budget_exhausted",
                "tokens_used": 0
            })

            continue

        # Call mock LLM with retry logic
        try:

            response = call_model_with_retry(
                ticket_id=ticket_id,
                prompt=prompt,
                redacted_text=row["redacted_text"]
            )

            # Extract response text
            response_text = response.choices[0].message.content

            # Add response token usage to running total
            tokens_used = response.usage.total_tokens
            total_tokens += tokens_used

            # Validate response
            validation = validate_response(response_text)

            # Store classification result
            results.append({
                "ticket_id": ticket_id,
                "customer_id": row["customer_id"],
                "true_category": row["true_category"],
                "category": validation["category"],
                "reason": validation["reason"],
                "status": validation["status"],
                "tokens_used": tokens_used
            })

        except TransientLLMError:

            results.append({
                "ticket_id": ticket_id,
                "customer_id": row["customer_id"],
                "true_category": row["true_category"],
                "category": "",
                "reason": "",
                "status": "model_error",
                "tokens_used": 0
            })

    return pd.DataFrame(results)

## 7.1 conclusion of the below line code 

The model used 502 tokens across six successful calls. Once the running total reached or exceeded 500, the function stopped making further model calls.

In [48]:
# Reset the mock LLM before this test
reset()

# Use a small temporary budget
test_budget = 500

# Run zero-shot classification
budget_test_results = run_classification_with_budget(
    ZERO_SHOT_PROMPT,
    test_budget
)

# Display results
print("Total tickets:", len(budget_test_results))

print("\nStatus counts:")
print(budget_test_results["status"].value_counts())

print("\nTotal tokens used:")
print(budget_test_results["tokens_used"].sum())

Total tickets: 164

Status counts:
status
budget_exhausted    150
classified_json       6
skipped_empty         4
flagged_long          4
Name: count, dtype: int64

Total tokens used:
502


## 7.2 Run the actual budget demonstration

**Use the Task 7th One def run_classification_with_budget funtion.**

In [49]:
TOKEN_BUDGET = 6000

reset()

budget_demo_results = run_classification_with_budget(
    ZERO_SHOT_PROMPT,
    TOKEN_BUDGET
)

print("Total tickets:", len(budget_demo_results))

print("\nStatus counts:")
print(budget_demo_results["status"].value_counts())

print("\nTotal tokens used:")
print(budget_demo_results["tokens_used"].sum())

print("\nBudget exhausted tickets:")
print(
    (budget_demo_results["status"] == "budget_exhausted").sum()
)

Ticket T0002: Retry 0, waiting 0.0454 seconds
Ticket T0116: Retry 0, waiting 0.0557 seconds
Ticket T0153: Retry 0, waiting 0.0478 seconds
Ticket T0039: Retry 0, waiting 0.0468 seconds
Ticket T0152: Retry 0, waiting 0.0572 seconds
Ticket T0043: Retry 0, waiting 0.0513 seconds
Ticket T0111: Retry 0, waiting 0.0530 seconds
Ticket T0155: Retry 0, waiting 0.0498 seconds
Ticket T0100: Retry 0, waiting 0.0527 seconds
Total tickets: 164

Status counts:
status
budget_exhausted          83
classified_json           64
classified_keyword         5
skipped_empty              4
flagged_long               4
flagged_invalid_output     4
Name: count, dtype: int64

Total tokens used:
6018

Budget exhausted tickets:
83


## 7.3 Save this one as a csv file 

In [50]:
# Save budget demonstration results
budget_demo_results.to_csv(
    OUTPUT_DIR / "triage_budget_demo.csv",
    index=False
)

print("Budget demonstration CSV saved successfully!")

print("File location:")
print(OUTPUT_DIR / "triage_budget_demo.csv")

Budget demonstration CSV saved successfully!
File location:
c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code\outputs\triage_budget_demo.csv


In [51]:
check_budget_file = pd.read_csv(
    OUTPUT_DIR / "triage_budget_demo.csv"
)

print("Rows:", len(check_budget_file))
display(check_budget_file.head())

Rows: 164


,ticket_id,customer_id,true_category,category,reason,status,tokens_used
0,T0024,C0486,fraud,fraud,keyword match,classified_json,86
1,T0073,C0080,billing,billing,keyword match,classified_json,82
2,T0095,C1087,technical,billing,keyword match,classified_json,84
3,T0159,C1172,none,NaN,NaN,skipped_empty,0
4,T0080,C0910,fraud,billing,keyword match,classified_json,86


## 7.4 Load the CSV file

In [52]:
# Load customer segmentation results from Part B

customer_segments = pd.read_csv(
    OUTPUT_DIR / "customer_segments.csv"
)

print("Customer segments loaded successfully!")

print("Total customers:", len(customer_segments))

print("\nColumn names:")
print(customer_segments.columns.tolist())

print("\nFirst 5 rows:")
display(customer_segments.head())

Customer segments loaded successfully!
Total customers: 1175

Column names:
['customer_id', 'recency', 'frequency', 'monetary', 'segment_id', 'segment_name', 'is_premium']

First 5 rows:


,customer_id,recency,frequency,monetary,segment_id,segment_name,is_premium
0,C0001,14,4,512.85,0,Occasional Customers,False
1,C0002,28,7,4644.90,1,Premium High-Spending Customers,True
2,C0003,7,22,3644.85,2,Frequent Loyal Customers,False
3,C0004,0,5,369.00,0,Occasional Customers,False
4,C0005,147,2,324.00,3,Dormant At-Risk Customers,False


## 7.5 Check the required columns

In [53]:
# Check whether required columns are available

required_columns = [
    "customer_id",
    "segment_name",
    "is_premium"
]

for column in required_columns:
    print(column, ":", column in customer_segments.columns)

customer_id : True
segment_name : True
is_premium : True


## 7.6 Check the premium values and data types

In [54]:
print("Data types:")
print(customer_segments[required_columns].dtypes)

print("\nPremium customer counts:")
print(customer_segments["is_premium"].value_counts(dropna=False))

Data types:
customer_id     object
segment_name    object
is_premium        bool
dtype: object

Premium customer counts:
is_premium
False    1037
True      138
Name: count, dtype: int64


## 7.7 Few-shot classification with budget 100000

In [55]:
# Set the final token budget
TOKEN_BUDGET = 100000

# Reset mock LLM counters before the full run
reset()

# Run final few-shot classification
final_few_shot_results = run_classification_with_budget(
    FEW_SHOT_PROMPT,
    TOKEN_BUDGET
)

print("Final few-shot classification completed!")

print("\nTotal tickets:", len(final_few_shot_results))

print("\nStatus counts:")
print(final_few_shot_results["status"].value_counts())

print("\nTotal tokens used:")
print(final_few_shot_results["tokens_used"].sum())

print("\nBudget exhausted tickets:")
print(
    (final_few_shot_results["status"] == "budget_exhausted").sum()
)

Ticket T0002: Retry 0, waiting 0.0491 seconds
Ticket T0116: Retry 0, waiting 0.0456 seconds
Ticket T0153: Retry 0, waiting 0.0546 seconds
Ticket T0039: Retry 0, waiting 0.0511 seconds
Ticket T0152: Retry 0, waiting 0.0555 seconds
Ticket T0043: Retry 0, waiting 0.0526 seconds
Ticket T0111: Retry 0, waiting 0.0555 seconds
Ticket T0155: Retry 0, waiting 0.0516 seconds
Ticket T0100: Retry 0, waiting 0.0544 seconds
Ticket T0068: Retry 0, waiting 0.0510 seconds
Ticket T0135: Retry 0, waiting 0.0557 seconds
Ticket T0009: Retry 0, waiting 0.0534 seconds
Ticket T0067: Retry 0, waiting 0.0570 seconds
Ticket T0106: Retry 0, waiting 0.0557 seconds
Ticket T0097: Retry 0, waiting 0.0497 seconds
Ticket T0003: Retry 0, waiting 0.0526 seconds
Ticket T0019: Retry 0, waiting 0.0450 seconds
Ticket T0086: Retry 0, waiting 0.0572 seconds
Ticket T0065: Retry 0, waiting 0.0468 seconds
Final few-shot classification completed!

Total tickets: 164

Status counts:
status
classified_json           135
classified_k

## 7.8 Merge the datasets

In [56]:
# Merge ticket classification results with customer segments

final_results = final_few_shot_results.merge(
    customer_segments[
        ["customer_id", "segment_name", "is_premium"]
    ],
    on="customer_id",
    how="left"
)

print("Datasets merged successfully!")

print("Total rows:", len(final_results))

print("\nFirst 5 rows:")
display(final_results.head())

Datasets merged successfully!
Total rows: 164

First 5 rows:


,ticket_id,customer_id,true_category,category,reason,status,tokens_used,segment_name,is_premium
0,T0024,C0486,fraud,fraud,keyword match,classified_json,181,Dormant At-Risk Customers,False
1,T0073,C0080,billing,billing,keyword match,classified_json,177,Frequent Loyal Customers,False
2,T0095,C1087,technical,technical,keyword match,classified_json,179,Occasional Customers,False
3,T0159,C1172,none,,,skipped_empty,0,Frequent Loyal Customers,False
4,T0080,C0910,fraud,fraud,keyword match,classified_json,181,Occasional Customers,False


## 7.9 Handle missing customer information

In [57]:
# Replace missing segment names with blank strings
final_results["segment_name"] = (
    final_results["segment_name"].fillna("")
)

# Treat missing premium status as False
final_results["is_premium"] = (
    final_results["is_premium"].fillna(False)
)

print("Missing segment names:",
      final_results["segment_name"].isna().sum())

print("\nMissing premium values:",
      final_results["is_premium"].isna().sum())

print("\nPremium status counts:")
print(final_results["is_premium"].value_counts())

Missing segment names: 0

Missing premium values: 0

Premium status counts:
is_premium
False    148
True      16
Name: count, dtype: int64


## 7.10 Create the routing logic

In [58]:
# Statuses that require human review
human_review_statuses = {
    "skipped_empty",
    "flagged_long",
    "flagged_invalid_output",
    "budget_exhausted"
}

# Define routing conditions in priority order
conditions = [
    final_results["status"].isin(human_review_statuses),

    final_results["category"].eq("fraud"),

    final_results["is_premium"]
]

# Define corresponding routes
choices = [
    "human_review",
    "security_team",
    "premium_desk"
]

# Assign routes
final_results["route"] = np.select(
    conditions,
    choices,
    default="support_queue"
)

# Assign priority
final_results["priority"] = np.where(
    final_results["route"].isin([
        "security_team",
        "premium_desk"
    ]),
    "HIGH",
    "NORMAL"
)

print("Ticket routing completed!")

Ticket routing completed!


## 7.11 Check the routing results

In [59]:
print("Route counts:")
print(final_results["route"].value_counts())

print("\nPriority counts:")
print(final_results["priority"].value_counts())

print("\nSample routed tickets:")
display(
    final_results[
        [
            "ticket_id",
            "customer_id",
            "status",
            "category",
            "segment_name",
            "is_premium",
            "priority",
            "route"
        ]
    ].head(10)
)

Route counts:
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64

Priority counts:
priority
NORMAL    119
HIGH       45
Name: count, dtype: int64

Sample routed tickets:


,ticket_id,customer_id,status,category,segment_name,is_premium,priority,route
0,T0024,C0486,classified_json,fraud,Dormant At-Risk Customers,False,HIGH,security_team
1,T0073,C0080,classified_json,billing,Frequent Loyal Customers,False,NORMAL,support_queue
2,T0095,C1087,classified_json,technical,Occasional Customers,False,NORMAL,support_queue
3,T0159,C1172,skipped_empty,,Frequent Loyal Customers,False,NORMAL,human_review
4,T0080,C0910,classified_json,fraud,Occasional Customers,False,HIGH,security_team
5,T0124,C0139,classified_json,fraud,Frequent Loyal Customers,False,HIGH,security_team
6,T0061,C0812,classified_json,billing,Frequent Loyal Customers,False,NORMAL,support_queue
7,T0118,C0355,classified_json,delivery,Frequent Loyal Customers,False,NORMAL,support_queue
8,T0037,C1156,classified_json,billing,Premium High-Spending Customers,True,HIGH,premium_desk
9,T0002,C0881,classified_json,delivery,Occasional Customers,False,NORMAL,support_queue


## 7.12 Create the final triage_results.csv

In [60]:
# Select the exact columns required by the assignment

final_columns = [
    "ticket_id",
    "customer_id",
    "status",
    "category",
    "segment_name",
    "priority",
    "route",
    "tokens_used"
]

triage_results = final_results[final_columns].copy()

print("Final columns selected successfully!")

print("\nColumn names:")
print(triage_results.columns.tolist())

print("\nFirst 10 rows:")
display(triage_results.head(10))

Final columns selected successfully!

Column names:
['ticket_id', 'customer_id', 'status', 'category', 'segment_name', 'priority', 'route', 'tokens_used']

First 10 rows:


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,Dormant At-Risk Customers,HIGH,security_team,181
1,T0073,C0080,classified_json,billing,Frequent Loyal Customers,NORMAL,support_queue,177
2,T0095,C1087,classified_json,technical,Occasional Customers,NORMAL,support_queue,179
3,T0159,C1172,skipped_empty,,Frequent Loyal Customers,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Occasional Customers,HIGH,security_team,181
5,T0124,C0139,classified_json,fraud,Frequent Loyal Customers,HIGH,security_team,181
6,T0061,C0812,classified_json,billing,Frequent Loyal Customers,NORMAL,support_queue,173
7,T0118,C0355,classified_json,delivery,Frequent Loyal Customers,NORMAL,support_queue,176
8,T0037,C1156,classified_json,billing,Premium High-Spending Customers,HIGH,premium_desk,177
9,T0002,C0881,classified_json,delivery,Occasional Customers,NORMAL,support_queue,175


## 7.13 Save the final CSV

In [61]:
# Save final ticket triage results

triage_results.to_csv(
    OUTPUT_DIR / "triage_results.csv",
    index=False
)

print("Final triage_results.csv created successfully!")

print("Saved location:")
print(OUTPUT_DIR / "triage_results.csv")

Final triage_results.csv created successfully!
Saved location:
c:\Users\PRAVEEN_KUMAR\jupyter_notebook\Jupyter notebook\Masai_Code\outputs\triage_results.csv


In [62]:
# Load the saved file again to verify it

saved_triage = pd.read_csv(
    OUTPUT_DIR / "triage_results.csv"
)

print("Total rows:", len(saved_triage))

print("\nTotal columns:", len(saved_triage.columns))

print("\nColumn names:")
print(saved_triage.columns.tolist())

print("\nStatus counts:")
print(saved_triage["status"].value_counts())

print("\nRoute counts:")
print(saved_triage["route"].value_counts())

display(saved_triage.head())

Total rows: 164

Total columns: 8

Column names:
['ticket_id', 'customer_id', 'status', 'category', 'segment_name', 'priority', 'route', 'tokens_used']

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
flagged_long                4
skipped_empty               4
Name: count, dtype: int64

Route counts:
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,Dormant At-Risk Customers,HIGH,security_team,181
1,T0073,C0080,classified_json,billing,Frequent Loyal Customers,NORMAL,support_queue,177
2,T0095,C1087,classified_json,technical,Occasional Customers,NORMAL,support_queue,179
3,T0159,C1172,skipped_empty,NaN,Frequent Loyal Customers,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,Occasional Customers,HIGH,security_team,181


In [63]:
# Final structural validation

assert len(triage_results) == 164

expected_columns = [
    "ticket_id",
    "customer_id",
    "status",
    "category",
    "segment_name",
    "priority",
    "route",
    "tokens_used"
]

assert triage_results.columns.tolist() == expected_columns

assert triage_results["ticket_id"].is_unique

print("Structure validation: PASSED")
print("Total rows:", len(triage_results))
print("Total columns:", len(triage_results.columns))

Structure validation: PASSED
Total rows: 164
Total columns: 8


In [64]:
# Verify human review routing

human_review_statuses = {
    "skipped_empty",
    "flagged_long",
    "flagged_invalid_output",
    "budget_exhausted"
}

human_mask = triage_results["status"].isin(
    human_review_statuses
)

assert (
    triage_results.loc[human_mask, "route"] == "human_review"
).all()

# Verify fraud routing
fraud_mask = (
    ~human_mask &
    triage_results["category"].eq("fraud")
)

assert (
    triage_results.loc[fraud_mask, "route"] == "security_team"
).all()

# Verify premium routing
premium_mask = (
    ~human_mask &
    ~fraud_mask &
    final_results["is_premium"]
)

assert (
    triage_results.loc[premium_mask, "route"] == "premium_desk"
).all()

print("Routing validation: PASSED")

Routing validation: PASSED


In [65]:
# Verify HIGH priority routes

high_routes = ["security_team", "premium_desk"]

high_mask = triage_results["route"].isin(high_routes)

assert (
    triage_results.loc[high_mask, "priority"] == "HIGH"
).all()

# All other routes should have NORMAL priority

normal_mask = ~high_mask

assert (
    triage_results.loc[normal_mask, "priority"] == "NORMAL"
).all()

print("Priority validation: PASSED")

Priority validation: PASSED


In [66]:
# Check that no original PII patterns remain in redacted text

patterns = [
    EMAIL_PATTERN,
    PHONE_PATTERN,
    ORDER_ID_PATTERN
]

for pattern in patterns:
    remaining = tickets_processed["redacted_text"].apply(
        lambda text: bool(re.search(pattern, text))
    ).sum()

    print("Remaining matches:", remaining)

    assert remaining == 0

print("\nPII redaction validation: PASSED")

Remaining matches: 0
Remaining matches: 0
Remaining matches: 0

PII redaction validation: PASSED


In [67]:
print("========== PART C FINAL VALIDATION ==========")

print("Rows:", len(triage_results))
print("Columns:", len(triage_results.columns))

print("\nStatus counts:")
print(triage_results["status"].value_counts())

print("\nRoute counts:")
print(triage_results["route"].value_counts())

print("\nTotal tokens used:")
print(triage_results["tokens_used"].sum())

print("\nAll final validation checks completed!")

========== PART C FINAL VALIDATION ==========
Rows: 164
Columns: 8

Status counts:
status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
flagged_long                4
skipped_empty               4
Name: count, dtype: int64

Route counts:
route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64

Total tokens used:
27652

All final validation checks completed!


# ----------- PART C Is Done ----------